# Gemma Guide Hackathon Demo Bootstrap

This notebook is the end-to-end bootstrap path for a fresh runtime.

It takes you from zero to a phone-openable HTTPS link by doing the following:

1. clone the repo
2. install dependencies
3. log into Hugging Face if needed
4. start the Gemma vLLM server
5. start `app_blind.py` in notebook/tunnel mode
6. expose the app through a Cloudflare Quick Tunnel
7. print an `https://...trycloudflare.com` link for your phone

The notebook is written to be usable on Kaggle, Colab, or a Linux GPU notebook VM.

Default demo profile locked in here:
- Gemma: `google/gemma-4-E4B-it`
- quantization: `bitsandbytes`
- served model name: `gemma-4-e4b-it`
- TIPS short-side: `672`
- target: fit and run end-to-end under a T4-class 16 GB budget

## Before you run it

- attach a GPU runtime
- make sure outbound internet is allowed
- accept the Hugging Face licenses for any Gemma / TIPS models you plan to use
- if you want a different model profile, change the config cell below; otherwise use the defaults as-is

## Runtime Notes

- The app itself runs on plain local HTTP inside the notebook VM.
- `cloudflared` provides the public HTTPS front door.
- Phone microphone access depends on opening the final `https://...trycloudflare.com` URL directly in the phone browser.
- If the runtime resets, rerun the notebook from the top.

In [ ]:
from pathlib import Path
import os
import platform

REPO_URL = "https://github.com/pariidanDKE/Gemma4-VisionAssistant.git"
REPO_REF = ""  # Optional branch, tag, or commit. Leave empty for default branch.

if Path('/kaggle/working').exists():
    REPO_DIR = Path('/kaggle/working/Gemma4-VisionAssistant')
elif Path('/content').exists():
    REPO_DIR = Path('/content/Gemma4-VisionAssistant')
else:
    REPO_DIR = Path.cwd() / 'Gemma4-VisionAssistant'

HF_TOKEN = os.getenv('HF_TOKEN', '')

# Demo profile: change these in one place if you want to test another serving setup.
VLLM_MODEL_REPO = "google/gemma-4-E4B-it"
VLLM_SERVED_NAME = "gemma-4-e4b-it"
VLLM_QUANTIZATION = "bitsandbytes"

# T4-oriented memory profile. Adjust only if you know why.
GPU_MEM_UTIL = "0.40"
MAX_MODEL_LEN = "4096"
MAX_NUM_SEQS = "1"
MAX_SOFT_TOKENS = "280"
TIPS_SHORT_SIDE = "672"

APP_HOST = "127.0.0.1"
APP_PORT = "7862"

print(f"Python: {platform.python_version()}")
print(f"Repo dir: {REPO_DIR}")
print(f"Serving model repo: {VLLM_MODEL_REPO}")
print(f"Served name: {VLLM_SERVED_NAME}")
print(f"TIPS short side: {TIPS_SHORT_SIDE}")

## Clone The Repo

If the repo already exists in the runtime, this cell leaves it in place.

In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=True)

## Install Dependencies

This installs the Python packages the current app path relies on, plus notebook-side helpers.

In [ ]:
%pip install -q -U pip
%pip install -q -r {REPO_DIR / 'requirements.txt'} fastapi uvicorn python-multipart pydub httpx huggingface_hub

In [ ]:
import shutil
import subprocess

if shutil.which('ffmpeg') is None:
    print('ffmpeg not found; attempting install...')
    subprocess.run(['bash', '-lc', 'apt-get update -y && apt-get install -y ffmpeg'], check=True)
else:
    print('ffmpeg already available')

## Hugging Face Login

Set `HF_TOKEN` in the runtime environment before running this cell, or paste it here directly if you are comfortable doing so in this notebook session.

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged into Hugging Face using HF_TOKEN from the environment.')
else:
    print('HF_TOKEN is empty. If model download fails, set HF_TOKEN and rerun this cell.')

## Start vLLM

This uses the repo's `scripts/start_gemma4.sh`, but drives it with explicit environment variables so the notebook is stable even if the repo defaults drift.

In [ ]:
import json
import sys
import time
import urllib.request
from pathlib import Path

LOG_DIR = Path('/tmp/gemma_demo_logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)
VLLM_LOG = LOG_DIR / 'vllm.log'

if 'vllm_proc' in globals() and vllm_proc.poll() is None:
    raise RuntimeError('vLLM appears to already be running in this notebook session.')

vllm_env = os.environ.copy()
vllm_env.update({
    'MODEL': VLLM_MODEL_REPO,
    'SERVED_NAME': VLLM_SERVED_NAME,
    'GPU_MEM_UTIL': GPU_MEM_UTIL,
    'MAX_MODEL_LEN': MAX_MODEL_LEN,
    'MAX_NUM_SEQS': MAX_NUM_SEQS,
    'MAX_SOFT_TOKENS': MAX_SOFT_TOKENS,
    'VLLM_QUANTIZATION': VLLM_QUANTIZATION,
})

with VLLM_LOG.open('w', encoding='utf-8') as log_file:
    vllm_proc = subprocess.Popen(
        ['bash', 'scripts/start_gemma4.sh'],
        cwd=str(REPO_DIR),
        env=vllm_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

print(f'Started vLLM with pid={vllm_proc.pid}. Waiting for /v1/models...')

deadline = time.time() + 900
while time.time() < deadline:
    if vllm_proc.poll() is not None:
        raise RuntimeError('vLLM exited early. Inspect /tmp/gemma_demo_logs/vllm.log')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=5) as resp:
            payload = json.loads(resp.read().decode())
            print(json.dumps(payload, indent=2)[:1200])
            break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError('Timed out waiting for vLLM to become ready.')

print('vLLM is ready.')

## Start `app_blind.py` In Notebook Mode

The app is bound to loopback and runs with `APP_DISABLE_SSL=1`. HTTPS is provided by the tunnel, not by uvicorn directly.

In [ ]:
APP_LOG = LOG_DIR / 'app_blind.log'

if 'app_proc' in globals() and app_proc.poll() is None:
    raise RuntimeError('app_blind.py appears to already be running in this notebook session.')

app_env = os.environ.copy()
app_env.update({
    'APP_HOST': APP_HOST,
    'APP_PORT': APP_PORT,
    'APP_DISABLE_SSL': '1',
    'VLLM_MODEL_ID': VLLM_SERVED_NAME,
    'SPATIALSENSE_TIPSV2_SHORT_SIDE': TIPS_SHORT_SIDE,
})

with APP_LOG.open('w', encoding='utf-8') as log_file:
    app_proc = subprocess.Popen(
        [sys.executable, 'app_blind.py'],
        cwd=str(REPO_DIR),
        env=app_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

print(f'Started app_blind.py with pid={app_proc.pid}. Waiting for local app...')

deadline = time.time() + 300
local_url = f'http://127.0.0.1:{APP_PORT}/'
while time.time() < deadline:
    if app_proc.poll() is not None:
        raise RuntimeError('app_blind.py exited early. Inspect /tmp/gemma_demo_logs/app_blind.log')
    try:
        with urllib.request.urlopen(local_url, timeout=5) as resp:
            html = resp.read().decode(errors='ignore')
            print(f'Local app responded with {len(html)} bytes')
            break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError('Timed out waiting for app_blind.py to become ready.')

print(f'Local app is ready on {local_url}')

## Download `cloudflared` And Open A Quick Tunnel

This creates the HTTPS URL you can open on your phone.

In [ ]:
import stat

if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'amd64'):
    raise RuntimeError(f'Expected Linux x86_64 runtime. Found: {platform.system()} {platform.machine()}')

cloudflared_path = LOG_DIR / 'cloudflared'
if not cloudflared_path.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path,
    )
    cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IEXEC)

print(f'cloudflared ready at {cloudflared_path}')

In [ ]:
import re

TUNNEL_LOG = LOG_DIR / 'cloudflared.log'

if 'cloudflared_proc' in globals() and cloudflared_proc.poll() is None:
    raise RuntimeError('cloudflared already appears to be running in this notebook session.')

tunnel_target = f'http://127.0.0.1:{APP_PORT}'
tunnel_log_file = TUNNEL_LOG.open('w', encoding='utf-8')
cloudflared_proc = subprocess.Popen(
    [str(cloudflared_path), 'tunnel', '--url', tunnel_target],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 90
recent = []

while time.time() < deadline:
    line = cloudflared_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    tunnel_log_file.write(line)
    tunnel_log_file.flush()
    recent.append(line.rstrip())
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError('Did not receive a public tunnel URL. Recent output:\n' + '\n'.join(recent[-20:]))

print('Open this URL on your phone:')
print(public_url)

## Phone Test Checklist

1. Open the printed `https://...trycloudflare.com` URL on your phone.
2. Allow microphone permission when prompted.
3. Take a photo or use the blind-first flow.
4. Ask a spoken question.
5. Confirm you receive both text and audio response.

If something breaks, inspect the log helper cells below.

## Log Helpers

In [ ]:
def tail_file(path, lines=80):
    path = Path(path)
    if not path.exists():
        print(f'{path} does not exist')
        return
    content = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    print('\n'.join(content[-lines:]))

print('tail_file(VLLM_LOG)')
print('tail_file(APP_LOG)')
print('tail_file(TUNNEL_LOG)')

## Cleanup

Run this when you are done with the demo session.

In [ ]:
for proc_name in ('cloudflared_proc', 'app_proc', 'vllm_proc'):
    proc = globals().get(proc_name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=20)
        print(f'Stopped {proc_name}')